# FlowCast (ICLR 2026) on INSAT-3S — VIS + WV

Drives the four-stage FlowCast pipeline on a Colab T4 GPU.

1. **Stage 1** — encode INSAT TIFFs with FlowCast's released SEVIR VAE (`b-rbmp/flowcast-cfm-sevir`)
2. **Stage 2** — sanity-check the VAE roundtrip (decision gate: PSNR ≥ 25 dB)
3. **Stage 3** — train the Earthformer-UNet I-CFM model (σ = 0.01, AdamW + cosine + EMA + FP16)
4. **Stage 4** — 48-frame forecast via 4 autoregressive blocks of 12, 8-sample pixel-space ensemble, threshold-based meteorological metrics (CRPS, CSI-M, FSS-M-P16, HSS-M, FAR-M)

Set the runtime to **GPU** (Runtime → Change runtime type → T4 GPU) before starting.

## Setup — mount Drive, cd, install deps (run after every fresh runtime)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/ISRO/ISRO A.1'   # <-- edit if your Drive path differs
%cd "$PROJECT_DIR"
!ls

In [ ]:
!pip install -q -r requirements.txt
!pip install -q imagecodecs huggingface_hub
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# ── Session status / resume check ──────────────────────────────────
# Run this after reconnecting to a Colab session to see what's already done.
import os, json, datetime as _dt

CHANNEL = 'vis'
cfg_map = {
    'vis': {'cfg': 'vis/config_fc.yaml', 'ckpt_dir': 'vis/checkpoints_fc', 'out_dir': 'vis/outputs_fc'},
    'wv':  {'cfg': 'wv/config_fc.yaml',  'ckpt_dir': 'wv/checkpoints_fc',  'out_dir': 'wv/outputs_fc'},
}

for ch, paths in cfg_map.items():
    print(f'\n── {ch.upper()} ──────────────────────────────────────────────')
    manifest = f'{ch}/manifest_fc.csv'
    print(f'  manifest        : {"✓ " + manifest if os.path.exists(manifest) else "✗ (run Stage 1)"}')
    ckpt = os.path.join(paths["ckpt_dir"], "best_flow.pt")
    if os.path.exists(ckpt):
        mtime = _dt.datetime.fromtimestamp(os.path.getmtime(ckpt)).strftime('%Y-%m-%d %H:%M')
        sz = os.path.getsize(ckpt) / 1e6
        hist = os.path.join(paths["ckpt_dir"], "flow_history.json")
        epochs_done = 0
        if os.path.exists(hist):
            h = json.load(open(hist))
            epochs_done = len(h.get("history", []))
        print(f'  best_flow.pt    : ✓ ({sz:.1f} MB, saved {mtime}, {epochs_done} epochs)')
    else:
        print(f'  best_flow.pt    : ✗ (run Stage 3)')
    sanity = os.path.join(paths["out_dir"], "sanity_metrics.json")
    if os.path.exists(sanity):
        m = json.load(open(sanity))
        print(f'  sanity_check    : ✓ PSNR={m["summary"]["psnr"]:.2f} dB  SSIM={m["summary"]["ssim"]:.3f}')
    else:
        print(f'  sanity_check    : ✗ (run Stage 2)')
    for tag in ['vae_decoder_ft_A.pt', 'vae_decoder_ft_B.pt']:
        p = os.path.join(paths["ckpt_dir"], tag)
        print(f'  {tag:<22}: {"✓ " + str(round(os.path.getsize(p)/1e6,1)) + " MB" if os.path.exists(p) else "✗ not trained yet"}')
    fc_outs = sorted(os.listdir(paths["out_dir"])) if os.path.isdir(paths["out_dir"]) else []
    metric_files = [f for f in fc_outs if f.endswith('_metrics.json') and 'flow_forecast' in f]
    print(f'  forecast outputs: {len(metric_files)} metrics file(s): {metric_files or "(none)"}')
print()


## VIS — Stage 1: encode TIFFs with the FlowCast VAE

Downloads the SEVIR autoencoder checkpoint once, then writes one `.pt` per frame to `vis/latents_fc/`.
Norm stats are computed on the first run and saved back into `vis/config_fc.yaml`.

In [ ]:
!python -m flowcast.encode_latents --config vis/config_fc.yaml

## VIS — Stage 2: VAE sanity check (decision gate)

The FlowCast VAE was trained on SEVIR radar, not INSAT — this is the load-bearing OOD check.

- **PSNR ≥ 25 dB**: proceed.
- **PSNR 20–25 dB**: proceed but expect a ceiling.
- **PSNR < 20 dB**: stop and decide (fall back to SD-VAE pipeline).

In [ ]:
from IPython.display import Image, display
!python -m flowcast.sanity_check --config vis/config_fc.yaml
display(Image('vis/outputs_fc/sanity_grid.png'))

In [ ]:
# ── Stage 2 gate check ──────────────────────────────────────────────
import json
m = json.load(open('vis/outputs_fc/sanity_metrics.json'))
psnr = m['summary']['psnr']
ssim = m['summary']['ssim']
print(f'VIS sanity — mean PSNR: {psnr:.2f} dB   SSIM: {ssim:.3f}')
if psnr >= 25:
    print('✅  GATE PASSED — proceed to Stage 3 (train_flow).')
elif psnr >= 20:
    print('⚠️  PSNR 20–25 dB — proceed, but forecast quality is ceiling-bounded by VAE reconstruction quality.')
else:
    raise AssertionError(
        f'GATE FAILED: PSNR={psnr:.2f} dB < 20 dB. '
        'The FlowCast VAE is very OOD for this channel. '
        'Consider the DiffCast pipeline (diffcast/colab_pipeline.ipynb) or the SD-VAE pipeline.')


## VIS — Stage 3: train the Earthformer-UNet I-CFM model

Paper-spec settings (T4-scaled): hidden=192, depth=4 cuboid blocks per stage, 4 attention heads,
lag=13 / lead=12, σ=0.01, AdamW lr=5e-4, cosine + 1% warmup, EMA decay 0.999, FP16.
Roughly 12–24 h on T4 across 2–3 sessions; the checkpoint is saved on every best-val improvement.

In [ ]:
!python -m flowcast.train_flow --config vis/config_fc.yaml

In [ ]:
# ── Training loss plot — run any time during or after Stage 3 ───────
import json, os
import matplotlib.pyplot as plt

hist_path = 'vis/checkpoints_fc/flow_history.json'
if not os.path.exists(hist_path):
    print('No history yet — run Stage 3 first.')
else:
    h = json.load(open(hist_path))
    history = h.get('history', [])
    epochs  = [e['epoch']      for e in history]
    tr_loss = [e['train_loss'] for e in history]
    va_loss = [e['val_loss']   for e in history]
    best_val = h.get('best_val', min(va_loss))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(epochs, tr_loss, label='train CFM loss')
    ax.plot(epochs, va_loss, label='val CFM loss', linewidth=2)
    ax.axhline(best_val, color='green', linestyle='--', alpha=0.7,
               label=f'best val = {best_val:.5f}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('CFM Loss (I-CFM, σ=0.01)')
    ax.set_title('FlowCast VIS — Training curve')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f'Epochs completed: {len(history)}   Best val loss: {best_val:.5f}')
    if history:
        last = history[-1]
        print(f'Last epoch {last["epoch"]}: train={last["train_loss"]:.5f}  val={last["val_loss"]:.5f}  lr={last["lr"]:.2e}')


## VIS — Stage 4: 48-frame forecast (single + 8-sample ensemble)

In [ ]:
!python -m flowcast.forecast_flow --config vis/config_fc.yaml --samples 1
!python -m flowcast.forecast_flow --config vis/config_fc.yaml --samples 8

In [ ]:
import glob, json
from IPython.display import Image, display

for p in sorted(glob.glob('vis/outputs_fc/flow_forecast_*_grid.png'))[-6:]:
    print(p); display(Image(p))

for p in sorted(glob.glob('vis/outputs_fc/flow_forecast_*_metrics.json')):
    m = json.load(open(p))
    print('\n', p, ' samples=', m.get('samples'), ' medoid_idx=', m.get('medoid_idx'))
    print('  thresholds:', m.get('thresholds'))
    sbm = m.get('summary_by_mode') or {'(legacy)': m.get('summary', {})}
    for mode, smry in sbm.items():
        print(f'  [{mode}]')
        for k, v in (smry or {}).items():
            print(f'    {k:>10}: {v:.4f}')
    if m.get('crps_mean') is not None:
        print(f'  [   crps]')
        print(f'    {"CRPS":>10}: {m["crps_mean"]:.4f}')


## VIS — Blur reduction (staged A → B + 3-way ensemble)

The forecast cells above produce **pixmean / medoid / latmean** trajectories side-by-side once `--samples > 1`. Below: fine-tune the VAE decoder on real frames (Strategy A), then on (predicted_latent → GT) pairs (Strategy B). Plan reference: `~/.claude/plans/s-whats-further-action-steady-clock.md`.

Phase numbers below match the plan.

### Phase 2 — Strategy A: decoder fine-tune on real (encode → decode) pairs

~45 min on T4. Freezes encoder, unfreezes decoder + post_quant_conv, optimises `0.1·MSE + 1.0·LPIPS-VGG`.
Saves checkpoint to `vis/checkpoints_fc/vae_decoder_ft_A.pt`.

In [ ]:
!pip install -q lpips==0.1.4

In [ ]:
!python -m flowcast.finetune_vae --config vis/config_fc.yaml

In [ ]:
# ── Strategy A — decoder fine-tune progress images ──────────────────
# Shows decoded vs ground-truth grids saved every 250 training steps.
import glob, os
from IPython.display import Image, display

prog_dir = 'vis/outputs_fc/vae_ft_A_progress'
imgs = sorted(glob.glob(os.path.join(prog_dir, '*.png')))
if not imgs:
    print('No progress images yet — run the finetune_vae cell first.')
else:
    # Show baseline (step 0) and last 3 snapshots
    show = ([imgs[0]] + imgs[-3:]) if len(imgs) > 3 else imgs
    for p in show:
        print(p)
        display(Image(p))


### Phase 3 — enable A + validate

Uncomments `vae.decoder_ckpt: vis/checkpoints_fc/vae_decoder_ft_A.pt` in the VIS config, then re-runs sanity_check (expect roundtrip PSNR +6 to +10 dB over baseline) and forecast (expect +0.5 to +1.5 dB on medoid).

In [ ]:
# enable A's decoder for inference
import re, pathlib
cfg_path = pathlib.Path('vis/config_fc.yaml')
text = cfg_path.read_text()
new_text = re.sub(
    r'^(\s*)#\s*decoder_ckpt:\s*vis/checkpoints_fc/vae_decoder_ft_A\.pt.*$',
    r'\1decoder_ckpt: vis/checkpoints_fc/vae_decoder_ft_A.pt',
    text, count=1, flags=re.MULTILINE)
if new_text == text:
    new_text = re.sub(
        r'^(\s*)decoder_ckpt:\s*vis/checkpoints_fc/vae_decoder_ft_B\.pt.*$',
        r'\1decoder_ckpt: vis/checkpoints_fc/vae_decoder_ft_A.pt',
        text, count=1, flags=re.MULTILINE)
cfg_path.write_text(new_text)
print('vae block now:')
for line in new_text.splitlines():
    if 'vae' in line.lower() or 'decoder_ckpt' in line:
        print(' ', line)


In [ ]:
!python -m flowcast.sanity_check --config vis/config_fc.yaml --num 30
!python -m flowcast.forecast_flow --config vis/config_fc.yaml --samples 8

### Phase 4 — build (predicted_latent → GT) pair dataset for Strategy B

~80 min on T4. Runs the CFM forecaster on every train-only day × 4 rollouts; saves predicted latents paired with GT metadata. Hard leakage asserts: no val/test UTCs touch the dataset.

In [ ]:
!python -m flowcast.build_pred_dataset --config vis/config_fc.yaml

### Phase 5 — Strategy B: decoder fine-tune on (pred_latent → GT) pairs

~100 min on T4. Decoder learns to map forecaster-output latents directly to sharp GT, compensating for both VAE OOD blur and forecaster mean-drift in one step. Inner train/val_holdout split is by-day (last 10% of training-pair days held out for early stop). Saves to `vis/checkpoints_fc/vae_decoder_ft_B.pt`.

In [ ]:
!python -m flowcast.finetune_vae_pred --config vis/config_fc.yaml

In [ ]:
# ── Strategy B — decoder fine-tune progress images ──────────────────
import glob, os
from IPython.display import Image, display

prog_dir = 'vis/outputs_fc/vae_ft_B_progress'
imgs = sorted(glob.glob(os.path.join(prog_dir, '*.png')))
if not imgs:
    print('No progress images yet — run the finetune_vae_pred cell first.')
else:
    show = ([imgs[0]] + imgs[-3:]) if len(imgs) > 3 else imgs
    for p in show:
        print(p)
        display(Image(p))

# Also print the training history summary
import json
hist_b = 'vis/checkpoints_fc/vae_decoder_ft_B_history.json'
if os.path.exists(hist_b):
    h = json.load(open(hist_b))
    baseline = h.get('baseline', {})
    best = h.get('best_val_psnr', 0)
    print(f'\nBaseline (SEVIR decoder) val PSNR : {baseline.get("val_psnr", "n/a"):.2f} dB')
    print(f'Best Strategy-B val PSNR          : {best:.2f} dB')
    print('Note: Strategy-B PSNR will be LOWER than Strategy-A on sanity_check.')
    print('      That is expected — the decoder is now specialised for forecaster latents.')
    print('      Watch the forecast metrics table below for the actual improvement.')


### Phase 6 — swap config to B + validate

Swaps `decoder_ckpt` from `_ft_A.pt` to `_ft_B.pt`. Note: sanity_check PSNR will DROP 1–3 dB vs A — the decoder is now specialised for forecaster latents, not encoder-mode latents. This is expected, not a regression. The signal that matters is the forecast metrics, especially medoid PSNR/SSIM/LPIPS.

In [ ]:
# swap A -> B
import re, pathlib
cfg_path = pathlib.Path('vis/config_fc.yaml')
text = cfg_path.read_text()
new_text = re.sub(
    r'^(\s*)decoder_ckpt:\s*vis/checkpoints_fc/vae_decoder_ft_A\.pt.*$',
    r'\1decoder_ckpt: vis/checkpoints_fc/vae_decoder_ft_B.pt',
    text, count=1, flags=re.MULTILINE)
cfg_path.write_text(new_text)
for line in new_text.splitlines():
    if 'decoder_ckpt' in line:
        print(' ', line)

In [ ]:
!python -m flowcast.sanity_check --config vis/config_fc.yaml --num 30
!python -m flowcast.forecast_flow --config vis/config_fc.yaml --samples 8

In [ ]:
# compare A vs B vs baseline (run the metrics-display cell above too)
import glob, json
all_runs = sorted(glob.glob('vis/outputs_fc/flow_forecast_*_metrics.json'))
print(f'{"file":<60s} {"mode":<10s} {"psnr":>7s} {"ssim":>7s} {"CSI_M":>7s}')
for p in all_runs[-3:]:
    m = json.load(open(p))
    name = p.split('/')[-1]
    sbm = m.get('summary_by_mode') or {'(legacy)': m.get('summary', {})}
    for mode, smry in sbm.items():
        psnr = smry.get('psnr', 0); ssim = smry.get('ssim', 0); csi = smry.get('CSI_M', 0)
        print(f'{name:<60s} {mode:<10s} {psnr:7.3f} {ssim:7.3f} {csi:7.3f}')
    if m.get('crps_mean') is not None:
        print(f'{"":<60s} {"crps":<10s} {m["crps_mean"]:7.4f}')

## WV — Stage 1 → 4 (same recipe, no day/night handling)

Don't kick this off until the VIS Stage-2 sanity check has cleared the decision gate above. WV is
easier (continuous 24 h signal) but pays cross-block drift over 4 rollout blocks.

In [ ]:
!python -m flowcast.encode_latents --config wv/config_fc.yaml
!python -m flowcast.sanity_check  --config wv/config_fc.yaml

In [ ]:
from IPython.display import Image, display
display(Image('wv/outputs_fc/sanity_grid.png'))

In [ ]:
!python -m flowcast.train_flow --config wv/config_fc.yaml

In [ ]:
!python -m flowcast.forecast_flow --config wv/config_fc.yaml --samples 1
!python -m flowcast.forecast_flow --config wv/config_fc.yaml --samples 8

In [ ]:
import glob, json
from IPython.display import Image, display

for p in sorted(glob.glob('wv/outputs_fc/flow_forecast_*_grid.png'))[-6:]:
    print(p); display(Image(p))

for p in sorted(glob.glob('wv/outputs_fc/flow_forecast_*_metrics.json')):
    m = json.load(open(p))
    print('\n', p, ' samples=', m.get('samples'), ' medoid_idx=', m.get('medoid_idx'))
    print('  thresholds:', m.get('thresholds'))
    sbm = m.get('summary_by_mode') or {'(legacy)': m.get('summary', {})}
    for mode, smry in sbm.items():
        print(f'  [{mode}]')
        for k, v in (smry or {}).items():
            print(f'    {k:>10}: {v:.4f}')
    if m.get('crps_mean') is not None:
        print(f'  [   crps]')
        print(f'    {"CRPS":>10}: {m["crps_mean"]:.4f}')


## WV — Blur Reduction (Strategy A and B)

Same strategy as VIS. Run only after WV Stage 4 forecast shows blurry results.
Strategy A fine-tunes the decoder on real WV encode-decode pairs (no risk to
the trained flow model). Strategy B builds a (pred_latent → GT) dataset from
the WV train split and trains the decoder on those pairs.

Note: WV typically needs less de-blurring than VIS because the SEVIR VAE
physics are closer to WV brightness temperature than to VIS reflectance.

In [ ]:
!pip install -q lpips==0.1.4

In [ ]:
# WV — Strategy A: decoder fine-tune on real encode-decode pairs (~45 min on T4)
!python -m flowcast.finetune_vae --config wv/config_fc.yaml

In [ ]:
# WV Strategy A — progress images
import glob, os
from IPython.display import Image, display

prog_dir = 'wv/outputs_fc/vae_ft_A_progress'
imgs = sorted(glob.glob(os.path.join(prog_dir, '*.png')))
if not imgs:
    print('No progress images — run the WV finetune_vae cell first.')
else:
    show = ([imgs[0]] + imgs[-3:]) if len(imgs) > 3 else imgs
    for p in show: print(p); display(Image(p))


### WV — Phase 3: enable Strategy A + validate

Uncomments `decoder_ckpt: wv/checkpoints_fc/vae_decoder_ft_A.pt` in the WV
config, then re-runs sanity_check and forecast to confirm the improvement.

In [ ]:
import re, pathlib
cfg_path = pathlib.Path('wv/config_fc.yaml')
text = cfg_path.read_text()
new_text = re.sub(
    r'^(\s*)#\s*decoder_ckpt:\s*wv/checkpoints_fc/vae_decoder_ft_A\.pt.*$',
    r'\1decoder_ckpt: wv/checkpoints_fc/vae_decoder_ft_A.pt',
    text, count=1, flags=re.MULTILINE)
if new_text == text:
    new_text = re.sub(
        r'^(\s*)decoder_ckpt:\s*wv/checkpoints_fc/vae_decoder_ft_B\.pt.*$',
        r'\1decoder_ckpt: wv/checkpoints_fc/vae_decoder_ft_A.pt',
        text, count=1, flags=re.MULTILINE)
cfg_path.write_text(new_text)
print('WV vae block now:')
for line in new_text.splitlines():
    if 'vae' in line.lower() or 'decoder_ckpt' in line:
        print(' ', line)


In [ ]:
!python -m flowcast.sanity_check  --config wv/config_fc.yaml --num 30
!python -m flowcast.forecast_flow --config wv/config_fc.yaml --samples 8

### WV — Phase 4: build (predicted_latent → GT) pair dataset (~80 min)

Runs the trained WV flow model on all training-split days to harvest
predicted latents paired with GT. Hard leakage check: val/test frames are
never included.

In [ ]:
!python -m flowcast.build_pred_dataset --config wv/config_fc.yaml

### WV — Phase 5: Strategy B decoder fine-tune (~100 min)

Trains the decoder on (pred_latent → GT) pairs harvested above. Saves to
`wv/checkpoints_fc/vae_decoder_ft_B.pt`.

In [ ]:
!python -m flowcast.finetune_vae_pred --config wv/config_fc.yaml

In [ ]:
import glob, os, json
from IPython.display import Image, display

prog_dir = 'wv/outputs_fc/vae_ft_B_progress'
imgs = sorted(glob.glob(os.path.join(prog_dir, '*.png')))
show = ([imgs[0]] + imgs[-3:]) if len(imgs) > 3 else imgs
for p in show: print(p); display(Image(p))

hist_b = 'wv/checkpoints_fc/vae_decoder_ft_B_history.json'
if os.path.exists(hist_b):
    h = json.load(open(hist_b))
    print(f'WV Strategy-B best val PSNR: {h.get("best_val_psnr", 0):.2f} dB')


### WV — Phase 6: swap config to Strategy B + validate

Swaps `decoder_ckpt` from `_ft_A.pt` → `_ft_B.pt`, then re-runs sanity_check
and forecast. Expect sanity PSNR to drop 1–3 dB vs A (the decoder is now
specialised for forecaster latents). The forecast metrics are what matters.

In [ ]:
import re, pathlib
cfg_path = pathlib.Path('wv/config_fc.yaml')
text = cfg_path.read_text()
new_text = re.sub(
    r'^(\s*)decoder_ckpt:\s*wv/checkpoints_fc/vae_decoder_ft_A\.pt.*$',
    r'\1decoder_ckpt: wv/checkpoints_fc/vae_decoder_ft_B.pt',
    text, count=1, flags=re.MULTILINE)
cfg_path.write_text(new_text)
for line in new_text.splitlines():
    if 'decoder_ckpt' in line: print(' ', line)


In [ ]:
!python -m flowcast.sanity_check  --config wv/config_fc.yaml --num 30
!python -m flowcast.forecast_flow --config wv/config_fc.yaml --samples 8

In [ ]:
# WV — compare baseline / A / B forecast metrics
import glob, json, os
print(f'{'file':<60s} {'mode':<10s} {'psnr':>7s} {'ssim':>7s} {'CSI_M':>7s}')
for p in sorted(glob.glob('wv/outputs_fc/flow_forecast_*_metrics.json')):
    m = json.load(open(p))
    name = p.split('/')[-1]
    sbm = m.get('summary_by_mode') or {'(legacy)': m.get('summary', {})}
    for mode, smry in sbm.items():
        psnr = smry.get('psnr', 0); ssim = smry.get('ssim', 0); csi = smry.get('CSI_M', 0)
        print(f'{name:<60s} {mode:<10s} {psnr:7.3f} {ssim:7.3f} {csi:7.3f}')
    if m.get('crps_mean') is not None:
        print(f'{"crps":<72s} {m["crps_mean"]:7.4f}')


## (Optional) Snapshot a channel's results to a shareable folder + zip

In [ ]:
import os, shutil, glob, json, zipfile, datetime as dt
import yaml

CH = 'vis'   # change to 'wv' for the WV snapshot
TS = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
SNAP = f'{CH}/results_snapshot_fc_{TS}'
os.makedirs(SNAP, exist_ok=True)

for p in sorted(glob.glob(f'{CH}/outputs_fc/*')):
    if os.path.isfile(p):
        shutil.copy2(p, os.path.join(SNAP, os.path.basename(p)))
for src in (f'{CH}/checkpoints_fc/best_flow.pt',
            f'{CH}/checkpoints_fc/flow_history.json',
            f'{CH}/checkpoints_fc/vae_decoder_ft_A.pt',
            f'{CH}/checkpoints_fc/vae_decoder_ft_A_history.json',
            f'{CH}/checkpoints_fc/vae_decoder_ft_B.pt',
            f'{CH}/checkpoints_fc/vae_decoder_ft_B_history.json'):
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(SNAP, os.path.basename(src)))
shutil.copy2(f'{CH}/config_fc.yaml', os.path.join(SNAP, 'config_fc.yaml'))

lines = [f'FlowCast {CH.upper()} snapshot — {TS}', '']
cfg = yaml.safe_load(open(f'{CH}/config_fc.yaml'))
lines += ['=== Channel ===',
          f'channel:    {cfg["channel"]}',
          f'source_dir: {cfg["source_dir"]}',
          f'norm:       low={cfg["norm"]["low"]}  high={cfg["norm"]["high"]}',
          f'vae.decoder_ckpt: {cfg.get("vae", {}).get("decoder_ckpt")}', '']
lines += ['=== Flow ===']
for k, v in cfg['flow'].items(): lines.append(f'  {k}: {v}')
lines += ['', '=== Forecast metrics ===']
for p in sorted(glob.glob(f'{CH}/outputs_fc/flow_forecast_*_metrics.json')):
    m = json.load(open(p))
    lines.append(f'\n[{os.path.basename(p)}]  samples={m.get("samples")}  medoid_idx={m.get("medoid_idx")}')
    lines.append(f'  thresholds: {m.get("thresholds")}')
    sbm = m.get('summary_by_mode') or {'(legacy)': m.get('summary', {})}
    for mode, smry in sbm.items():
        lines.append(f'  [{mode}]')
        for k, v in (smry or {}).items():
            lines.append(f'    {k:>10}: {v:.4f}')
    if m.get('crps_mean') is not None:
        lines.append(f'  [crps]')
        lines.append(f'    {"CRPS":>10}: {m["crps_mean"]:.4f}')
with open(os.path.join(SNAP, 'README.txt'), 'w') as f:
    f.write('\n'.join(lines))

zip_path = f'{CH}/results_snapshot_fc_{TS}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir(SNAP)):
        zf.write(os.path.join(SNAP, fn), arcname=os.path.join(os.path.basename(SNAP), fn))
print('Snapshot:', SNAP)
print('Zip:     ', zip_path)